
# 02 — Baseline, contact point e primo fit Hertz-like

Questo notebook parte dagli output già generati dal reader:

```text
afm_output/
├── csv_curves/
├── summary_segment0_with_qc.csv
└── plots_overview/
```

Obiettivo: prendere le curve già convertite in CSV, stimare baseline e contact point, costruire una indentazione relativa e fare un primo fit meccanico:

\[
F(\delta) = A\,\delta^{3/2}
\]

Per ora `A` è un **parametro elastico efficace**, non ancora il modulo di Young assoluto. Per convertirlo in \(E\) servono geometria della punta, raggio e coefficiente di Poisson.


In [ ]:

from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.optimize import curve_fit
from scipy.signal import savgol_filter

warnings.filterwarnings("ignore")

PROJECT_DIR = Path.cwd()
OUTPUT_DIR = PROJECT_DIR / "afm_output"
CSV_DIR = OUTPUT_DIR / "csv_curves"
SUMMARY_PATH = OUTPUT_DIR / "summary_segment0_with_qc.csv"

FIT_OUTPUT_DIR = OUTPUT_DIR / "fit_hertz_like"
FIT_PLOTS_DIR = FIT_OUTPUT_DIR / "plots"
FIT_TABLE_PATH = FIT_OUTPUT_DIR / "fit_results_segment0.csv"

FIT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIT_PLOTS_DIR.mkdir(parents=True, exist_ok=True)

print("Working directory:", PROJECT_DIR)
print("CSV_DIR exists:", CSV_DIR.exists(), CSV_DIR)
print("SUMMARY exists:", SUMMARY_PATH.exists(), SUMMARY_PATH)



## 1. Caricamento del summary

Il summary contiene una riga per curva. Lo useremo per sapere quali curve esistono, a quale gruppo appartengono e quali erano state eventualmente flaggate nel controllo qualità preliminare.


In [ ]:

summary = pd.read_csv(SUMMARY_PATH)
print("Numero righe summary:", len(summary))
display(summary.head())

print("\nColonne disponibili:")
print(list(summary.columns))


## 2. Trova i CSV delle curve

Il notebook cerca automaticamente i CSV del segmento 0, cioè quello che stiamo trattando come approach/extend.

Nota importante: il notebook 01 salva i CSV in sottocartelle per gruppo. Per questo qui usiamo `CSV_DIR.rglob("*.csv")`, cioè una ricerca ricorsiva.


In [ ]:
# Cerca ricorsivamente: i CSV del notebook 01 sono dentro sottocartelle per gruppo.
all_csv = sorted(CSV_DIR.rglob("*.csv"))
print("CSV trovati:", len(all_csv))
print("Primi 20 CSV:")
for p in all_csv[:20]:
    print(" -", p.relative_to(CSV_DIR))

# Segmento 0 = extend/approach. Pattern robusto sui nomi tipo cito-0007_segment0.csv.
seg0_csv = [
    p for p in all_csv
    if (
        "segment0" in p.name.lower()
        or "segment_0" in p.name.lower()
        or "seg0" in p.name.lower()
        or "seg_0" in p.name.lower()
    )
]

# Fallback: se il nome non contiene segment0, prova a leggere meta_segment_id.
if len(seg0_csv) == 0 and len(all_csv) > 0:
    print("\nNon trovo segment0 nel nome. Provo a leggere meta_segment_id dentro i CSV...")
    candidate = []
    for p in all_csv:
        try:
            tmp = pd.read_csv(p, nrows=1)
            if "meta_segment_id" in tmp.columns and int(tmp["meta_segment_id"].iloc[0]) == 0:
                candidate.append(p)
        except Exception:
            pass
    seg0_csv = candidate

print("\nCSV segmento 0 selezionati:", len(seg0_csv))

if len(seg0_csv) == 0:
    raise RuntimeError(
        "Non ho trovato CSV del segmento 0. "
        "Controlla che il notebook 01 abbia esportato i CSV e che siano in afm_output/csv_curves/."
    )



## 3. Funzioni: pulizia, contact point e fit

La logica è volutamente semplice e robusta:

1. prende `z_tip_um` e `force_nN`;
2. sottrae la baseline stimata sulla parte iniziale della curva;
3. liscia leggermente la forza per stimare il punto di contatto;
4. trova il primo punto in cui la forza supera una soglia basata sul rumore;
5. definisce l'indentazione relativa come distanza dal contact point;
6. fitta la regione post-contatto con \(F = A \delta^{3/2}\).

Questa è una prima versione. Non pretende ancora di sostituire il workflow manuale.


In [ ]:
def sanitize_name(s):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(s))


def hertz_like(delta_um, A):
    """F [nN] = A * delta_um^(3/2)."""
    delta_um = np.asarray(delta_um)
    return A * np.power(np.clip(delta_um, 0, None), 1.5)


def get_columns(df):
    """Trova colonne principali anche se il reader ha nomi leggermente diversi."""
    z_candidates = ["z_tip_um", "verticalTipPosition_um", "z_um"]
    f_candidates = ["force_nN", "force_corr_nN", "Force_nN"]

    z_col = next((c for c in z_candidates if c in df.columns), None)
    f_col = next((c for c in f_candidates if c in df.columns), None)

    if z_col is None:
        z_like = [c for c in df.columns if ("z" in c.lower() or "position" in c.lower()) and "um" in c.lower()]
        z_col = z_like[0] if z_like else None

    if f_col is None:
        f_like = [c for c in df.columns if "force" in c.lower() and "nn" in c.lower()]
        f_col = f_like[0] if f_like else None

    if z_col is None or f_col is None:
        raise ValueError(f"Non trovo colonne z/force. Colonne disponibili: {list(df.columns)}")

    return z_col, f_col


def robust_edge_baseline(force, frac=0.20):
    """
    Baseline robusta scegliendo una delle due estremità della curva.

    Nei tuoi plot il non-contatto è spesso a destra, mentre il contatto/indentazione è a sinistra.
    Questa funzione non assume a priori il verso: confronta le due estremità e sceglie quella
    con forza mediana più vicina a zero.
    """
    f = np.asarray(force, dtype=float)
    n = len(f)
    nbase = max(50, int(frac * n))

    left = f[:nbase]
    right = f[-nbase:]

    left_med = np.nanmedian(left)
    right_med = np.nanmedian(right)

    left_noise = 1.4826 * np.nanmedian(np.abs(left - left_med))
    right_noise = 1.4826 * np.nanmedian(np.abs(right - right_med))

    if not np.isfinite(left_noise) or left_noise == 0:
        left_noise = np.nanstd(left)
    if not np.isfinite(right_noise) or right_noise == 0:
        right_noise = np.nanstd(right)

    if abs(right_med) <= abs(left_med):
        return right_med, right_noise, "right"
    return left_med, left_noise, "left"


def smooth_force(force):
    n = len(force)
    win = min(101, max(11, (n // 50) | 1))
    if win >= n:
        win = max(5, (n // 5) | 1)
    try:
        return savgol_filter(force, window_length=win, polyorder=2)
    except Exception:
        return force.copy()


def fit_hertz_like_curve(csv_path, make_plot=True):
    df = pd.read_csv(csv_path)
    z_col, f_col = get_columns(df)

    z = df[z_col].to_numpy(dtype=float)
    f = df[f_col].to_numpy(dtype=float)

    good = np.isfinite(z) & np.isfinite(f)
    z = z[good]
    f = f[good]

    if len(z) < 200:
        raise ValueError("Troppi pochi punti utili.")

    # Ordiniamo per z crescente. Questo rende il fit indipendente dall'ordine temporale dei punti nel CSV.
    order = np.argsort(z)
    z = z[order]
    f = f[order]

    baseline, noise, baseline_side = robust_edge_baseline(f, frac=0.20)
    fcorr = f - baseline
    f_smooth = smooth_force(fcorr)

    # Soglia positiva per individuare l'uscita dalla baseline.
    max_force = np.nanmax(fcorr)
    threshold = max(5.0 * noise, 0.03 * max_force)
    min_consecutive = 20
    above = f_smooth > threshold

    if baseline_side == "right":
        # Non-contatto a destra, indentazione a sinistra.
        # Cerco il punto di contatto scorrendo da destra verso sinistra.
        contact_idx = None
        for i in range(len(z) - min_consecutive - 1, min_consecutive, -1):
            if np.all(above[i - min_consecutive:i]):
                contact_idx = i
                break

        if contact_idx is None:
            grad = np.gradient(f_smooth, z)
            search_start = int(0.05 * len(z))
            search_end = int(0.95 * len(z))
            contact_idx = search_start + int(np.nanargmax(np.abs(grad[search_start:search_end])))

        z0 = z[contact_idx]
        delta = z0 - z  # positiva andando verso sinistra

    else:
        # Non-contatto a sinistra, indentazione a destra.
        contact_idx = None
        for i in range(0, len(z) - min_consecutive):
            if np.all(above[i:i + min_consecutive]):
                contact_idx = i
                break

        if contact_idx is None:
            grad = np.gradient(f_smooth, z)
            search_start = int(0.05 * len(z))
            search_end = int(0.95 * len(z))
            contact_idx = search_start + int(np.nanargmax(np.abs(grad[search_start:search_end])))

        z0 = z[contact_idx]
        delta = z - z0  # positiva andando verso destra

    # Punti post-contatto: indentazione positiva e forza sopra soglia.
    fit_region_mask = (
        np.isfinite(delta)
        & np.isfinite(fcorr)
        & (delta > 0)
        & (fcorr > threshold)
    )

    dfit_all = delta[fit_region_mask]
    ffit_all = fcorr[fit_region_mask]

    if len(dfit_all) < 80:
        raise ValueError("Fit region insufficiente.")

    # Evito primi punti incerti e parte estrema, spesso più soggetta a non-linearità/saturazione.
    dmin = np.nanquantile(dfit_all, 0.05)
    dmax = np.nanquantile(dfit_all, 0.85)
    fit_window = (dfit_all >= dmin) & (dfit_all <= dmax)

    dfit = dfit_all[fit_window]
    ffit = ffit_all[fit_window]

    if len(dfit) < 80:
        dfit = dfit_all
        ffit = ffit_all

    p0 = [np.nanmax(ffit) / max(np.nanmax(dfit) ** 1.5, 1e-9)]

    popt, pcov = curve_fit(
        hertz_like,
        dfit,
        ffit,
        p0=p0,
        bounds=(0, np.inf),
        maxfev=20000,
    )

    A = popt[0]
    A_err = float(np.sqrt(pcov[0, 0])) if pcov is not None and np.isfinite(pcov[0, 0]) else np.nan

    pred = hertz_like(dfit, A)
    resid = ffit - pred
    rss = float(np.nansum(resid**2))
    tss = float(np.nansum((ffit - np.nanmean(ffit))**2))
    r2 = 1.0 - rss / tss if tss > 0 else np.nan
    rmse = float(np.sqrt(np.nanmean(resid**2)))

    result = {
        "csv_file": csv_path.name,
        "csv_path": str(csv_path),
        "n_points": len(z),
        "baseline_side": baseline_side,
        "z_min_um": float(np.nanmin(z)),
        "z_max_um": float(np.nanmax(z)),
        "baseline_nN": float(baseline),
        "baseline_noise_pN": float(noise * 1000.0),
        "contact_idx": int(contact_idx),
        "contact_z_um": float(z0),
        "threshold_nN": float(threshold),
        "n_fit_points": int(len(dfit)),
        "delta_fit_min_um": float(np.nanmin(dfit)),
        "delta_fit_max_um": float(np.nanmax(dfit)),
        "A_nN_per_um_3over2": float(A),
        "A_err_nN_per_um_3over2": float(A_err),
        "fit_rmse_nN": rmse,
        "fit_r2": float(r2),
        "fit_status": "ok",
    }

    if make_plot:
        fig, ax = plt.subplots(figsize=(7.5, 5))

        ax.plot(z, fcorr, lw=0.8, alpha=0.7, label="curve baseline-corrected")
        ax.plot(z, f_smooth, lw=1.0, alpha=0.9, label="smoothed")
        ax.axvline(z0, ls="--", lw=1.2, label="estimated contact")
        ax.axhline(threshold, ls=":", lw=1.0, label="threshold")

        if baseline_side == "right":
            zfit = z0 - dfit
            dgrid = np.linspace(np.nanmin(dfit), np.nanmax(dfit), 300)
            zgrid = z0 - dgrid
        else:
            zfit = z0 + dfit
            dgrid = np.linspace(np.nanmin(dfit), np.nanmax(dfit), 300)
            zgrid = z0 + dgrid

        ax.scatter(zfit, ffit, s=4, alpha=0.35, label="fit points")
        ax.plot(zgrid, hertz_like(dgrid, A), lw=2, label=r"fit $A\delta^{3/2}$")

        ax.set_xlabel("Vertical tip position [µm]")
        ax.set_ylabel("Force - baseline [nN]")
        ax.set_title(f"{csv_path.stem} | baseline: {baseline_side}")
        ax.grid(alpha=0.3)
        ax.legend(fontsize=8)
        fig.tight_layout()

        out_png = FIT_PLOTS_DIR / f"{sanitize_name(csv_path.stem)}__fit.png"
        fig.savefig(out_png, dpi=160, bbox_inches="tight")
        plt.close(fig)

        result["fit_plot"] = str(out_png)

    return result



## 4. Test su una curva singola

Prima di lanciare il fit su tutte le curve, guardiamo un singolo caso.


In [ ]:

test_csv = seg0_csv[0]
print("Test CSV:", test_csv)

res = fit_hertz_like_curve(test_csv, make_plot=True)
res


In [ ]:

from IPython.display import Image, display

display(Image(filename=res["fit_plot"]))



## 5. Fit batch su tutte le curve segmento 0

Questa cella può richiedere un po' di tempo. Produce:

```text
afm_output/fit_hertz_like/fit_results_segment0.csv
afm_output/fit_hertz_like/plots/*.png
```


In [ ]:

results = []
failures = []

for i, csv_path in enumerate(seg0_csv, start=1):
    try:
        r = fit_hertz_like_curve(csv_path, make_plot=True)
        results.append(r)
    except Exception as e:
        failures.append({"csv_file": csv_path.name, "csv_path": str(csv_path), "fit_status": "failed", "error": str(e)})

    if i % 25 == 0 or i == len(seg0_csv):
        print(f"Processed {i}/{len(seg0_csv)}")

fit_results = pd.DataFrame(results + failures)
fit_results.to_csv(FIT_TABLE_PATH, index=False)

print("Fit ok:", len(results))
print("Fit failed:", len(failures))
print("Saved:", FIT_TABLE_PATH)

display(fit_results.head())



## 6. Recupera gruppo sperimentale dal nome/percorso

Se il CSV contiene già `meta_group`, usiamo quello. Altrimenti cerchiamo di ricostruirlo dal nome del file/percorso.


In [ ]:

def infer_group_from_csv(csv_path):
    try:
        df0 = pd.read_csv(csv_path, nrows=1)
        if "meta_group" in df0.columns:
            return str(df0["meta_group"].iloc[0])
    except Exception:
        pass

    s = str(csv_path)
    known_groups = summary["group"].dropna().unique() if "group" in summary.columns else []
    for g in known_groups:
        if str(g) in s:
            return str(g)
    return "unknown"

fit_results["group"] = fit_results["csv_path"].apply(infer_group_from_csv)
fit_results.to_csv(FIT_TABLE_PATH, index=False)

display(fit_results[["csv_file", "group", "fit_status", "A_nN_per_um_3over2", "A_err_nN_per_um_3over2", "fit_r2", "fit_rmse_nN"]].head())



## 7. Primo controllo dei risultati

Questi grafici non sono ancora conclusivi: servono a capire se il fit sta lavorando in modo ragionevole e se alcuni gruppi hanno distribuzioni molto diverse del parametro efficace \(A\).


In [ ]:

ok = fit_results[fit_results["fit_status"] == "ok"].copy()

print("Fit validi:", len(ok))
display(ok[["A_nN_per_um_3over2", "A_err_nN_per_um_3over2", "fit_r2", "fit_rmse_nN", "baseline_noise_pN"]].describe())

plt.figure(figsize=(7, 4.5))
plt.hist(ok["A_nN_per_um_3over2"].replace([np.inf, -np.inf], np.nan).dropna(), bins=30)
plt.xlabel(r"A [nN / µm$^{3/2}$]")
plt.ylabel("Conteggi")
plt.title("Distribuzione parametro Hertz-like efficace")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIT_OUTPUT_DIR / "hist_A_hertz_like.png", dpi=160, bbox_inches="tight")
plt.show()

groups = [g for g in ok["group"].dropna().unique()]
data = [ok.loc[ok["group"] == g, "A_nN_per_um_3over2"].dropna().values for g in groups]

plt.figure(figsize=(10, 5))
plt.boxplot(data, labels=groups, showfliers=True)
plt.xticks(rotation=30, ha="right")
plt.ylabel(r"A [nN / µm$^{3/2}$]")
plt.title("Parametro Hertz-like efficace per gruppo")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(FIT_OUTPUT_DIR / "boxplot_A_by_group.png", dpi=160, bbox_inches="tight")
plt.show()



## 8. Curve con fit sospetto

Guardiamo le curve con fit peggiore. Tipicamente sono quelle con \(R^2\) basso, RMSE alto o parametro \(A\) fuori scala.


In [ ]:

bad_fit = ok.sort_values(["fit_r2", "fit_rmse_nN"], ascending=[True, False]).head(20)
display(bad_fit[["csv_file", "group", "A_nN_per_um_3over2", "fit_r2", "fit_rmse_nN", "baseline_noise_pN", "fit_plot"]])



## 9. Interpretazione provvisoria

Cose da controllare nei plot salvati:

- il contact point stimato cade dove inizia davvero la crescita della forza?
- il fit usa una regione sensata?
- il segno della forza è corretto?
- ci sono curve in cui l'algoritmo fitta il tratto sbagliato?
- l'ordinamento di `z` è coerente con l'approach?

Questa versione serve per individuare i problemi. La prossima iterazione migliorerà il contact point e la scelta automatica del fit range.
